In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [8]:
# ============================================
# TITANIC LOGISTIC REGRESSION - IMPROVED JAX
# ============================================

# -----------------------------
# IMPORTS
# -----------------------------
import pandas as pd
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import time
import numpy as np
from sklearn.model_selection import train_test_split

# -----------------------------
# MEMBER 1: DATA LOADING & PREPROCESSING
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")

# --- Extract titles from names ---
def get_title(name):
    title = name.split(',')[1].split('.')[0].strip()
    # Map to common titles
    if title in ['Mr']:
        return 'Mr'
    elif title in ['Mrs', 'Ms']:
        return 'Mrs'
    elif title in ['Miss', 'Mlle']:
        return 'Miss'
    elif title in ['Master']:
        return 'Master'
    else:
        return 'Other'
train['Title'] = train['Name'].apply(get_title)

# --- Fill missing values ---
# Age: median by Pclass + Title (more accurate)
train['Age'] = train.groupby(['Pclass', 'Title'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
train['Age'] = train['Age'].fillna(train['Age'].median())  # fallback

# Fare: median
train['Fare'] = train['Fare'].fillna(train['Fare'].median())

# Embarked: mode
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# Drop irrelevant columns (keep Name? we already used it, drop now)
train = train.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# --- New features ---
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

# --- One-hot encode categoricals ---
categorical_cols = ['Sex', 'Embarked', 'Title']
train = pd.get_dummies(train, columns=categorical_cols, drop_first=True)

# --- Define feature columns dynamically ---
# Numerical columns we want to normalize
numerical_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
# Binary columns: everything else except the target 'Survived'
all_features = [col for col in train.columns if col != 'Survived']
# Separate numerical and binary features for clarity
binary_features = [col for col in all_features if col not in numerical_features]

# Store the full feature list (order matters for test alignment)
features = numerical_features + binary_features

X = train[features].values.astype(jnp.float32)
y = train['Survived'].values.reshape(-1, 1).astype(jnp.float32)

# --- Normalize numerical features only ---
X_num = X[:, :len(numerical_features)]
X_mean = X_num.mean(axis=0)
X_std = X_num.std(axis=0) + 1e-8
X_num_norm = (X_num - X_mean) / X_std
X = jnp.concatenate([X_num_norm, X[:, len(numerical_features):]], axis=1)

# Add a column of ones for bias
X = jnp.concatenate([jnp.ones((X.shape[0], 1)), X], axis=1)

y = jnp.array(y)
print("Data preprocessing done. X shape:", X.shape, "y shape:", y.shape)

# --- Split into train / validation sets ---
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Validation size: {X_val.shape[0]}")

# -----------------------------
# MEMBER 2: MODEL DEFINITION
# -----------------------------
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict_single(w, x):
    return sigmoid(jnp.dot(x, w))

predict_batch = vmap(predict_single, in_axes=(None, 0))

def loss_fn(w, X, y, lambda_reg=0.01):
    preds = predict_batch(w, X).reshape(-1, 1)
    # Cross-entropy loss
    ce = -jnp.mean(y * jnp.log(preds + 1e-8) + (1 - y) * jnp.log(1 - preds + 1e-8))
    # L2 regularization (exclude bias term w[0])
    reg = lambda_reg * 0.5 * jnp.sum(w[1:] ** 2)
    return ce + reg

# -----------------------------
# MEMBER 3: TRAINING WITH ADAM + MINI-BATCHES + EARLY STOPPING
# -----------------------------
def adam_update(w, m, v, t, grad, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * (grad ** 2)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    w = w - lr * m_hat / (jnp.sqrt(v_hat) + eps)
    return w, m, v

grad_fn = grad(loss_fn)

@jit
def update_step(w, m, v, t, X_batch, y_batch, lr):
    g = grad_fn(w, X_batch, y_batch)
    return adam_update(w, m, v, t, g, lr)

def train_adam(X, y, X_val, y_val, lr=0.001, batch_size=32, epochs=100,
               lambda_reg=0.01, patience=10, verbose=True):
    n_samples = X.shape[0]
    n_features = X.shape[1]
    key = jax.random.PRNGKey(0)
    # Heuristic initialization (small random)
    w = jax.random.normal(key, (n_features, 1)) * 0.01
    m = jnp.zeros_like(w)
    v = jnp.zeros_like(w)

    best_val_loss = float('inf')
    best_w = w
    wait = 0
    t = 0  # step counter

    for epoch in range(epochs):
        # Shuffle data at the beginning of each epoch
        perm = jax.random.permutation(key, n_samples)
        X_shuffled = X[perm]
        y_shuffled = y[perm]

        epoch_loss = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            if X_batch.shape[0] < batch_size:  # drop last incomplete batch
                continue
            t += 1
            w, m, v = update_step(w, m, v, t, X_batch, y_batch, lr)
            # accumulate loss for reporting
            batch_loss = loss_fn(w, X_batch, y_batch, lambda_reg)
            epoch_loss += batch_loss
            n_batches += 1

        epoch_loss /= n_batches
        val_loss = loss_fn(w, X_val, y_val, lambda_reg)

        if verbose:
            print(f"Epoch {epoch+1:3d} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_w = w
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    return best_w

# --- Train model ---
start = time.time()
w = train_adam(X_train, y_train, X_val, y_val,
               lr=0.01, batch_size=64, epochs=200, lambda_reg=0.005, patience=10)
end = time.time()
print(f"Training finished in {end - start:.2f} seconds")

# --- Compute final training and validation accuracy ---
train_preds = predict_batch(w, X_train).reshape(-1, 1)
train_acc = jnp.mean((train_preds > 0.5).astype(int) == y_train)
val_preds = predict_batch(w, X_val).reshape(-1, 1)
val_acc = jnp.mean((val_preds > 0.5).astype(int) == y_val)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

# -----------------------------
# MEMBER 4: TEST DATA PREPROCESSING & PREDICTION
# -----------------------------
test = pd.read_csv(path + "test.csv")
test_ids = test['PassengerId']

# --- Apply same preprocessing to test data ---
# Extract titles
test['Title'] = test['Name'].apply(get_title)

# Fill missing values using training medians (by Pclass + Title for Age)
# Use the same grouping as in training
test['Age'] = test.groupby(['Pclass', 'Title'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
test['Age'] = test['Age'].fillna(train['Age'].median())
test['Fare'] = test['Fare'].fillna(train['Fare'].median())

# Drop irrelevant columns
test = test.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# New features
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

# One-hot encoding
categorical_cols_test = ['Sex', 'Embarked', 'Title']
test = pd.get_dummies(test, columns=categorical_cols_test, drop_first=True)

# Align test columns with training features
# Add any missing columns (with zeros) and reorder to match training
for col in features:
    if col not in test.columns:
        test[col] = 0
test = test[features]  # same order as training

# Convert to float32
X_test = test.values.astype(jnp.float32)

# Normalize numerical features using training means/stds
X_test_num = X_test[:, :len(numerical_features)]
X_test_num_norm = (X_test_num - X_mean) / X_std
X_test = jnp.concatenate([X_test_num_norm, X_test[:, len(numerical_features):]], axis=1)

# Add bias column
X_test = jnp.concatenate([jnp.ones((X_test.shape[0], 1)), X_test], axis=1)

# Predict
test_preds = predict_batch(w, X_test).reshape(-1, 1)
test_preds_binary = (test_preds > 0.5).astype(int)

# Create submission
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': test_preds_binary.flatten()
})
submission.to_csv('submission.csv', index=False)
print("Submission created ✅")
print(submission.head())

# -----------------------------
# REFLECTION ANSWERS
# -----------------------------
print("\nReflection:")
print("1️⃣ vmap is more efficient than looping over individual samples because it vectorizes the prediction computation across all samples in a batch using XLA. This eliminates Python overhead and enables efficient parallel execution on accelerators.")
print("2️⃣ Combining jit with grad improves training speed by compiling the entire gradient update step into optimized machine code. jit reduces the overhead of Python function calls and allows the JAX compiler to fuse operations, resulting in faster iterations and better convergence due to more stable numeric optimizations.")

Data preprocessing done. X shape: (891, 15) y shape: (891, 1)
Train size: 712, Validation size: 179
Epoch   1 | Train Loss: 0.6493 | Val Loss: 0.6098
Epoch   2 | Train Loss: 0.5766 | Val Loss: 0.5568
Epoch   3 | Train Loss: 0.5380 | Val Loss: 0.5236
Epoch   4 | Train Loss: 0.5144 | Val Loss: 0.4989
Epoch   5 | Train Loss: 0.4974 | Val Loss: 0.4799
Epoch   6 | Train Loss: 0.4848 | Val Loss: 0.4656
Epoch   7 | Train Loss: 0.4754 | Val Loss: 0.4551
Epoch   8 | Train Loss: 0.4681 | Val Loss: 0.4473
Epoch   9 | Train Loss: 0.4625 | Val Loss: 0.4415
Epoch  10 | Train Loss: 0.4580 | Val Loss: 0.4371
Epoch  11 | Train Loss: 0.4545 | Val Loss: 0.4338
Epoch  12 | Train Loss: 0.4517 | Val Loss: 0.4312
Epoch  13 | Train Loss: 0.4494 | Val Loss: 0.4292
Epoch  14 | Train Loss: 0.4475 | Val Loss: 0.4277
Epoch  15 | Train Loss: 0.4460 | Val Loss: 0.4264
Epoch  16 | Train Loss: 0.4447 | Val Loss: 0.4254
Epoch  17 | Train Loss: 0.4436 | Val Loss: 0.4245
Epoch  18 | Train Loss: 0.4427 | Val Loss: 0.4239
